In [1]:
# Import necessary libraries
import cv2
import os
from keras.applications import VGG16
from keras.models import Model
from keras.layers import Dense, Input, Flatten, Dropout
from keras.optimizers import Adam
import numpy as np
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

In [2]:
# Define paths for your dataset of eye and mouth images
open_eye_dir = 'C:/ALL/Project Final year/eye and mouth ML HAAR/dataset_new/train/open_eyes'
closed_eye_dir = 'C:/ALL/Project Final year/eye and mouth ML HAAR/dataset_new/train/closed_eyes'
mouth_open_dir = 'C:/ALL/Project Final year/eye and mouth ML HAAR/dataset_new/train/yawn'
mouth_closed_dir = 'C:/ALL/Project Final year/eye and mouth ML HAAR/dataset_new/train/no_yawn'

In [3]:

# Prepare lists to store images and labels
eye_images = []
eye_labels = []
mouth_images = []
mouth_labels = []

In [4]:
# Load eye images and assign labels
for file in os.listdir(open_eye_dir):
    img = cv2.imread(os.path.join(open_eye_dir, file), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = cv2.resize(img, (224, 224))  # VGG16 requires larger image size
        eye_images.append(img)
        eye_labels.append(1)  # Label for open eyes

for file in os.listdir(closed_eye_dir):
    img = cv2.imread(os.path.join(closed_eye_dir, file), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = cv2.resize(img, (224, 224))
        eye_images.append(img)
        eye_labels.append(0)  # Label for closed eyes

In [5]:
# Load mouth images and assign labels
for file in os.listdir(mouth_open_dir):
    img = cv2.imread(os.path.join(mouth_open_dir, file), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = cv2.resize(img, (224, 224))
        mouth_images.append(img)
        mouth_labels.append(1)  # Label for yawning

for file in os.listdir(mouth_closed_dir):
    img = cv2.imread(os.path.join(mouth_closed_dir, file), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = cv2.resize(img, (224, 224))
        mouth_images.append(img)
        mouth_labels.append(0)  # Label for no yawning

In [6]:
# Convert to numpy arrays
eye_images = np.array(eye_images).reshape(-1, 224, 224, 1)
mouth_images = np.array(mouth_images).reshape(-1, 224, 224, 1)
eye_labels = np.array(eye_labels)
mouth_labels = np.array(mouth_labels)

In [7]:
# Normalize the images
eye_images = eye_images / 255.0
mouth_images = mouth_images / 255.0

In [8]:
# One-hot encode the labels (binary classification)
eye_labels = to_categorical(eye_labels, num_classes=2)
mouth_labels = to_categorical(mouth_labels, num_classes=2)

In [9]:
# Split data into training and test sets
X_train_eye, X_test_eye, y_train_eye, y_test_eye = train_test_split(eye_images, eye_labels, test_size=0.2, random_state=42)
X_train_mouth, X_test_mouth, y_train_mouth, y_test_mouth = train_test_split(mouth_images, mouth_labels, test_size=0.2, random_state=42)

In [10]:
# Define the input layer
input_layer_eye = Input(shape=(224, 224, 1))
input_layer_mouth = Input(shape=(224, 224, 1))

In [11]:
# Use VGG16 for eye detection
vgg_eye = VGG16(include_top=False, weights=None, input_tensor=input_layer_eye)
eye_flatten = Flatten()(vgg_eye.output)
eye_output = Dense(2, activation='softmax', name='eye_output')(eye_flatten)

# Use VGG16 for mouth detection
vgg_mouth = VGG16(include_top=False, weights=None, input_tensor=input_layer_mouth)
mouth_flatten = Flatten()(vgg_mouth.output)
mouth_output = Dense(2, activation='softmax', name='mouth_output')(mouth_flatten)


In [12]:
# Define the model with two inputs and two outputs
model = Model(inputs=[input_layer_eye, input_layer_mouth], outputs=[eye_output, mouth_output])

ValueError: The name "block1_conv1" is used 2 times in the model. All layer names should be unique.

In [ ]:
#Train the model
epochs = 30
history = model.fit(
    [X_train_eye, X_train_mouth], [y_train_eye, y_train_mouth],
    epochs=epochs,
    validation_data=([X_test_eye, X_test_mouth], [y_test_eye, y_test_mouth]),
    batch_size=32
)

In [ ]:
# Plot accuracy and loss graph
plt.figure(figsize=(12, 6))
plt.plot(history.history['eye_output_accuracy'], label='Eye Train Accuracy')
plt.plot(history.history['val_eye_output_accuracy'], label='Eye Validation Accuracy')
plt.plot(history.history['mouth_output_accuracy'], label='Mouth Train Accuracy')
plt.plot(history.history['val_mouth_output_accuracy'], label='Mouth Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='upper left')
plt.show()

# Optionally, plot the loss as well
plt.figure(figsize=(12, 6))
plt.plot(history.history['eye_output_loss'], label='Eye Train Loss')
plt.plot(history.history['val_eye_output_loss'], label='Eye Validation Loss')
plt.plot(history.history['mouth_output_loss'], label='Mouth Train Loss')
plt.plot(history.history['val_mouth_output_loss'], label='Mouth Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper left')
plt.show()

In [ ]:
# Predict on test set
y_pred_eye = model.predict(X_test_eye)
y_pred_eye_class = np.argmax(y_pred_eye, axis=1)
y_test_eye_class = np.argmax(y_test_eye, axis=1)

In [ ]:
# Calculate the confusion matrix for eye detection
conf_mat_eye = confusion_matrix(y_test_eye_class, y_pred_eye_class)
print("Eye Confusion Matrix:")
print(conf_mat_eye)


In [ ]:
# Predict on mouth test set
y_pred_mouth = model.predict(X_test_mouth)
y_pred_mouth_class = np.argmax(y_pred_mouth, axis=1)
y_test_mouth_class = np.argmax(y_test_mouth, axis=1)

In [ ]:
# Calculate the confusion matrix for mouth detection
conf_mat_mouth = confusion_matrix(y_test_mouth_class, y_pred_mouth_class)
print("Mouth Confusion Matrix:")
print(conf_mat_mouth)


In [ ]:
# Save the model
model.save('models/eye_and_mouth_model_vggnet.h5')